In [107]:
#!python -m venv venv

In [108]:
#!venv\Scripts\activate

!ollama pull phi3:mini
!ollama pull qwen2.5-coder:7b

In [109]:
#!ollama pull deepseek-llm

In [110]:
#!ollama pull deepseek-coder:6.7b

In [111]:
#!ollama pull gemma2:2b

In [112]:
#!ollama pull mistral:7b

In [113]:
#!ollama list

installer dépendences

In [114]:
#%pip install pandas sqlalchemy pymysql requests sentence-transformers faiss-cpu sqlglot

import requests
import pandas as pd
import numpy as np
import faiss
import re
import time

from sentence_transformers import SentenceTransformer

In [115]:
import pymysql
print("PyMySQL installed")

PyMySQL installed


change le chemain après: (normal terminal)
    cd C:\Users\Nouhaila\Desktop\test_db-master
    C:\xampp\mysql\bin\mysql.exe -u root employee
    source employees.sql;
FOR TEST : 
    SHOW TABLES;
    SELECT COUNT(*) FROM employees;

Connecté Mysql

In [116]:
from sqlalchemy import create_engine
import pandas as pd

DATABASE_URL = (
    "mysql+pymysql://root:@localhost:3306/employees"
)

engine = create_engine(DATABASE_URL)

print("MySQL connected")

MySQL connected


Auto schéma extracteur

shema text

Database TEST

##########################################################
##########################################################
##########################################################
##########################################################

In [117]:
user_question = "Show departments with their average salary"

In [118]:
tables_query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = DATABASE();
"""

tables_df = pd.read_sql(
    tables_query,
    engine
)

print(tables_df)

             table_name
0      current_dept_emp
1           departments
2              dept_emp
3  dept_emp_latest_date
4          dept_manager
5             employees
6              salaries
7                titles


In [119]:
schema_metadata = []

for table in tables_df["table_name"]:

    columns_query = f"""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = DATABASE()
    AND table_name = '{table}';
    """

    columns_df = pd.read_sql(
        columns_query,
        engine
    )

    schema_metadata.append({
        "table": table,
        "columns": columns_df[
            "column_name"
        ].tolist()
    })

print(schema_metadata)

[{'table': 'current_dept_emp', 'columns': ['emp_no', 'dept_no', 'from_date', 'to_date']}, {'table': 'departments', 'columns': ['dept_no', 'dept_name']}, {'table': 'dept_emp', 'columns': ['emp_no', 'dept_no', 'from_date', 'to_date']}, {'table': 'dept_emp_latest_date', 'columns': ['emp_no', 'from_date', 'to_date']}, {'table': 'dept_manager', 'columns': ['emp_no', 'dept_no', 'from_date', 'to_date']}, {'table': 'employees', 'columns': ['emp_no', 'birth_date', 'first_name', 'last_name', 'gender', 'hire_date']}, {'table': 'salaries', 'columns': ['emp_no', 'salary', 'from_date', 'to_date']}, {'table': 'titles', 'columns': ['emp_no', 'title', 'from_date', 'to_date']}]


In [120]:
relationships_query = """
SELECT
    TABLE_NAME,
    COLUMN_NAME,
    REFERENCED_TABLE_NAME,
    REFERENCED_COLUMN_NAME
FROM information_schema.KEY_COLUMN_USAGE
WHERE REFERENCED_TABLE_NAME IS NOT NULL
AND TABLE_SCHEMA = DATABASE();
"""

relationships_df = pd.read_sql(
    relationships_query,
    engine
)

print(relationships_df)

     TABLE_NAME COLUMN_NAME REFERENCED_TABLE_NAME REFERENCED_COLUMN_NAME
0      dept_emp      emp_no             employees                 emp_no
1      dept_emp     dept_no           departments                dept_no
2  dept_manager      emp_no             employees                 emp_no
3  dept_manager     dept_no           departments                dept_no
4      salaries      emp_no             employees                 emp_no
5        titles      emp_no             employees                 emp_no


In [121]:
table_semantics = []

for item in schema_metadata:

    table_name = item["table"]

    columns = item["columns"]

    semantic_prompt = f"""
You are a database analyst.

Understand the purpose of this SQL table.

Table:
{table_name}

Columns:
{', '.join(columns)}

Explain:
- what the table probably represents
- what kind of records it stores

Keep answer short.
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "mistral:7b",
            "prompt": semantic_prompt,
            "stream": False,
            "temperature": 0,
            "num_predict": 40
        }
    )

    semantic_text = response.json()[
        "response"
    ].strip()

    table_semantics.append({
        "table": table_name,
        "semantic": semantic_text
    })

print(table_semantics)

[{'table': 'current_dept_emp', 'semantic': 'This SQL table, `current_dept_emp`, likely represents the current employment status of employees in different departments, with their respective department numbers, start dates (`from_date`), and end dates (`to_date`). It probably stores records indicating which employees are currently assigned to which departments as of the current date, if the end date is null or not specified.'}, {'table': 'departments', 'semantic': 'This SQL table, `departments`, likely represents a database structure for storing information about different departments within an organization. The table stores records containing the department number (`dept_no`) and corresponding department name.'}, {'table': 'dept_emp', 'semantic': 'The `dept_emp` table likely represents the employment history of employees within different departments in an organization. It stores records of when employees joined (from_date) and left (to_date) each department, identified by dept_no, and t

In [122]:
relationship_semantics = []

for rel in relationships_df.itertuples():

    prompt = f"""
You are analyzing SQL database relationships.

Source table:
{rel.TABLE_NAME}

Source column:
{rel.COLUMN_NAME}

Target table:
{rel.REFERENCED_TABLE_NAME}

Target column:
{rel.REFERENCED_COLUMN_NAME}

Explain:
- what this relationship probably represents
- how the tables are connected
- what type of entities this relationship links

Keep answer short.
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "mistral:7b",
            "prompt": prompt,
            "stream": False,
            "temperature": 0,
            "num_predict": 50
        }
    )

    explanation = response.json()[
        "response"
    ].strip()

    relationship_semantics.append(
        explanation
    )

print(relationship_semantics[:5])

['This relationship likely represents a one-to-one mapping between the `dept_emp` and `employees` tables, where each employee record in the `employees` table corresponds to exactly one row in the `dept_emp` table, and vice versa. The entities this relationship links are employees within their respective departments.', 'This relationship likely represents the association between department employees (dept_emp) and departments themselves (departments). The tables are connected via a common column, dept_no, which links the department number in both tables. This relationship links department entities with their respective employee details.', "This relationship probably represents a one-to-many association, where each department manager (dept_manager) is managing multiple employees (employees). The connection between the tables is established via the common emp_no column. Therefore, this relationship links department managers as the 'one' entity and employees as the 'many' entities.", 'This

In [123]:
schema_texts = []

for item in schema_metadata:

    table_name = item["table"]

    columns = item["columns"]

    semantic_description = ""

    for semantic in table_semantics:

        if semantic["table"] == table_name:

            semantic_description = semantic[
                "semantic"
            ]

    text = f"""
Table: {table_name}

Meaning:
{semantic_description}

Columns:
{chr(10).join('- ' + c for c in columns)}
"""

    schema_texts.append(text)

print(schema_texts[0])


Table: current_dept_emp

Meaning:
This SQL table, `current_dept_emp`, likely represents the current employment status of employees in different departments, with their respective department numbers, start dates (`from_date`), and end dates (`to_date`). It probably stores records indicating which employees are currently assigned to which departments as of the current date, if the end date is null or not specified.

Columns:
- emp_no
- dept_no
- from_date
- to_date



In [124]:
schema_embeddings = embedding_model.encode(
    schema_texts
).astype("float32")

print(schema_embeddings.shape)

(8, 384)


In [125]:
dimension = schema_embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(schema_embeddings)

print("FAISS ready")

FAISS ready


In [126]:
query_embedding = embedding_model.encode(
    [user_question]
).astype("float32")

D, I = index.search(
    query_embedding,
    k=3
)

pre_normalization_schema = ""

for idx in I[0]:

    pre_normalization_schema += (
        schema_texts[idx]
    )

    pre_normalization_schema += "\n"

print(pre_normalization_schema)


Table: salaries

Meaning:
The `salaries` table likely represents employee salary information and stores records of each employee's salary details along with their effective dates, such as start date and end date (or current status) for a specific salary period.

Columns:
- emp_no
- salary
- from_date
- to_date


Table: departments

Meaning:
This SQL table, `departments`, likely represents a database structure for storing information about different departments within an organization. The table stores records containing the department number (`dept_no`) and corresponding department name.

Columns:
- dept_no
- dept_name


Table: dept_emp

Meaning:
The `dept_emp` table likely represents the employment history of employees within different departments in an organization. It stores records of when employees joined (from_date) and left (to_date) each department, identified by dept_no, and their unique employee number, emp_no.

Columns:
- emp_no
- dept_no
- from_date
- to_date




In [127]:
relationship_text = ""

for _, row in relationships_df.iterrows():

    relationship_text += f"""
{row['TABLE_NAME']}.{row['COLUMN_NAME']}
references
{row['REFERENCED_TABLE_NAME']}.{row['REFERENCED_COLUMN_NAME']}
"""

print(relationship_text)


dept_emp.emp_no
references
employees.emp_no

dept_emp.dept_no
references
departments.dept_no

dept_manager.emp_no
references
employees.emp_no

dept_manager.dept_no
references
departments.dept_no

salaries.emp_no
references
employees.emp_no

titles.emp_no
references
employees.emp_no



In [128]:
intent_prompt = f"""
You rewrite database requests clearly.

DATABASE CONTEXT:
{pre_normalization_schema}

DATABASE RELATIONSHIPS:
{relationship_text}

TASK:
Rewrite the request clearly while preserving EXACT meaning.

STRICT RULES:
- Rewrite ONLY in natural language
- NEVER generate SQL
- NEVER explain
- NEVER answer
- Preserve ALL logic
- Preserve ALL conditions
- Preserve historical meaning
- Preserve aggregation meaning
- Use database vocabulary when relevant
- Keep sentence short

IMPORTANT:
- average must stay average
- count must stay count
- max must stay max
- min must stay min
- never changed must keep same meaning

User Request:
{user_question}

Rewritten Request:
"""

In [129]:
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "mistral:7b",
        "prompt": intent_prompt,
        "stream": False,
        "temperature": 0,
        "num_predict": 30
    }
)

normalized_question = response.json()[
    "response"
].strip()

print(normalized_question)

Display the list of departments and their corresponding average salaries.


In [130]:
query_embedding = embedding_model.encode(
    [normalized_question]
).astype("float32")

D, I = index.search(
    query_embedding,
    k=3
)

In [131]:
relevant_semantics = []

for idx in I[0]:

    relevant_semantics.append(
        table_semantics[idx]["semantic"]
    )

semantic_context = "\n".join(
    relevant_semantics
)

print(semantic_context)

The `salaries` table likely represents employee salary information and stores records of each employee's salary details along with their effective dates, such as start date and end date (or current status) for a specific salary period.
This SQL table, `departments`, likely represents a database structure for storing information about different departments within an organization. The table stores records containing the department number (`dept_no`) and corresponding department name.
The `dept_emp` table likely represents the employment history of employees within different departments in an organization. It stores records of when employees joined (from_date) and left (to_date) each department, identified by dept_no, and their unique employee number, emp_no.


In [132]:
expanded_indices = set(I[0])

retrieved_tables = []

for idx in I[0]:

    table_name = schema_metadata[idx]["table"]

    retrieved_tables.append(
        table_name
    )

for _, row in relationships_df.iterrows():

    source_table = row[
        "TABLE_NAME"
    ]

    target_table = row[
        "REFERENCED_TABLE_NAME"
    ]

    if source_table in retrieved_tables:

        for i, item in enumerate(
            schema_metadata
        ):

            if item["table"] == target_table:

                expanded_indices.add(i)

    if target_table in retrieved_tables:

        for i, item in enumerate(
            schema_metadata
        ):

            if item["table"] == source_table:

                expanded_indices.add(i)

print(expanded_indices)

{1, 2, 4, 5, 6}


In [133]:
mini_schema = ""

for idx in expanded_indices:

    mini_schema += schema_texts[idx]

    mini_schema += "\n"

print(mini_schema)


Table: departments

Meaning:
This SQL table, `departments`, likely represents a database structure for storing information about different departments within an organization. The table stores records containing the department number (`dept_no`) and corresponding department name.

Columns:
- dept_no
- dept_name


Table: dept_emp

Meaning:
The `dept_emp` table likely represents the employment history of employees within different departments in an organization. It stores records of when employees joined (from_date) and left (to_date) each department, identified by dept_no, and their unique employee number, emp_no.

Columns:
- emp_no
- dept_no
- from_date
- to_date


Table: dept_manager

Meaning:
The `dept_manager` SQL table likely represents a record of department managers within an organization. It stores records that include manager's employee number (emp_no), the corresponding department number (dept_no), and the date range during which they served as a manager (from_date, to_date).


In [134]:
sql_prompt = f"""
You are an expert MySQL SQL generator.

DATABASE SCHEMA:
{mini_schema}

DATABASE RELATIONSHIPS:
{relationship_text}

DATABASE SEMANTICS:
{semantic_context}

IMPORTANT RULES:
- Return ONLY SQL
- No markdown
- No explanation
- Use ONLY existing tables
- Use ONLY existing columns
- Never invent columns
- Never invent tables
- Use valid MySQL syntax
- Keep query simple
- Use LIMIT 10 unless aggregation is required

Question:
{normalized_question}

SQL:
"""

In [135]:
start = time.time()

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "deepseek-coder:6.7b",
        "prompt": sql_prompt,
        "stream": False,
        "temperature": 0,
        "num_predict": 150
    }
)

end = time.time()

sql_query = response.json()[
    "response"
]

print(f"Generation time: {end - start:.2f} sec")

Generation time: 153.97 sec


In [136]:
sql_query = re.sub(
    r"```sql",
    "",
    sql_query
)

sql_query = re.sub(
    r"```",
    "",
    sql_query
)

sql_query = re.sub(
    r"<\\|.*?\\|>",
    "",
    sql_query
)

sql_query = re.sub(
    r"▁",
    "",
    sql_query
)

sql_query = re.sub(
    r"\s+",
    " ",
    sql_query
)

sql_query = sql_query.strip()

sql_query = sql_query.split(";")[0] + ";"

print(sql_query)

SELECT d.dept_name, AVG(s.salary) as avg_salary FROM departments d JOIN dept_emp de ON d.dept_no = de.dept_no JOIN salaries s ON de.emp_no = s.emp_no GROUP BY d.dept_name;
